In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option("display.max_columns", None)
DATA_PATH = "../data/processed/bank_clean.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)

(10000, 17)


In [3]:
orders = {
    "AgeBand": ["18-29", "30-39", "40-49", "50-59", "60+"],
    "BalanceBand": ["Zero", "Low", "Mid", "High"],
    "ProductGroup": ["1", "2", "3+"],
}
for col, order in orders.items():
    df[col] = pd.Categorical(df[col].astype(str), categories=order, ordered=True)
print(df[list(orders)].isna().sum())


AgeBand         0
BalanceBand     0
ProductGroup    0
dtype: int64


In [4]:
df["HighValue"] = (df["BalanceBand"] == "High").astype(int)
df["HighValue"] = ((df["Balance"]>= 100_000) & (df["IsActiveMember"] == 1)).astype(int)

In [5]:
df["HighValue_Active"] = ((df["Balance"] >= 100_000) & (df["IsActiveMember"] == 1)).astype(int)
print(df.shape)  

(10000, 19)


In [6]:
def value_summary(maks, name):
    sub = df[maks]
    churned = sub[sub["Exited"] == 1]
    return {
         "segment": name,
        "customers": len(sub),
        "churned": len(churned),
        "churn_rate": round(len(churned) / len(sub), 4),
        "avg_balance": round(sub["Balance"].mean(), 0),
        "balance_total": round(sub["Balance"].sum(), 0),
        "balance_leaving": round(churned["Balance"].sum(), 0),
    }
summary = pd.DataFrame([
    value_summary(df["Exited"].isin([0,1]), "All customers"),
    value_summary(df["HighValue"] == 1, "High balance band"),
    value_summary(df["HighValue_Active"] == 1, "Balance >=100k & active"),
    
])
summary["pct_of_all_balance_leaving"] = (
    summary["balance_leaving"] / summary.loc[0, "balance_leaving"]
).round(4)
summary

,segment,customers,churned,churn_rate,avg_balance,balance_total,balance_leaving,pct_of_all_balance_leaving
0,All customers,10000,2037,0.2037,76486.0,764858893.0,185588095.0,1.0000
1,High balance band,2443,439,0.1797,132247.0,323079431.0,57850796.0,0.3117
2,Balance >=100k & active,2443,439,0.1797,132247.0,323079431.0,57850796.0,0.3117


In [7]:
print(df.shape)  
print([c for c in df.columns if "High" in c])


(10000, 19)
['HighValue', 'HighValue_Active']


In [8]:
print(df["HighValue"].sum(), df["HighValue_Active"].sum())
print((df["HighValue"] == df["HighValue_Active"]).all())
print(df.loc[df["HighValue"] == 1, "BalanceBand"].value_counts())

2443 2443
True
BalanceBand
Mid     1084
High    1076
Low      283
Zero       0
Name: count, dtype: int64


In [9]:
df["HighValue"] = (df["BalanceBand"] == "High").astype(int)

print(df["HighValue"].sum())  
print(df.loc[df["HighValue"] == 1, "BalanceBand"].value_counts())  
print((df["HighValue"] == df["HighValue_Active"]).all())  

2128
BalanceBand
High    2128
Zero       0
Low        0
Mid        0
Name: count, dtype: int64
False


In [10]:
hv = df[df["HighValue"] == 1]
total_hv_churners = hv["Exited"].sum()

def hv_profile(col):
     g = hv.groupby(col, observed=True).agg(
        customers=("Exited", "size"),
        churned=("Exited", "sum"),
        balance_leaving=("Balance", lambda s: s[hv.loc[s.index, "Exited"] == 1].sum()),
    )
     g["churn_rate"] = (g["churned"] / g["customers"]).round(4)
     g["share_of_hv_churners"] = (g["churned"] / total_hv_churners).round(4)
     g["share_of_balance_leaving"] = (g["balance_leaving"] / hv.loc[hv["Exited"] == 1, "Balance"].sum()).round(4)
     return g

for c in ["Geography","AgeBand","ProductGroup","IsActiveMember","Gender"]:
    print(f"\n---{col}---")
    display(hv_profile(c))


---ProductGroup---


,customers,churned,balance_leaving,churn_rate,share_of_hv_churners,share_of_balance_leaving
Geography,,,,,,
France,905,186,29566459.11,0.2055,0.3773,0.3901
Germany,785,218,31619552.39,0.2777,0.4422,0.4172
Spain,438,89,14609325.89,0.2032,0.1805,0.1927



---ProductGroup---


,customers,churned,balance_leaving,churn_rate,share_of_hv_churners,share_of_balance_leaving
AgeBand,,,,,,
18-29,356,39,6111240.32,0.1096,0.0791,0.0806
30-39,902,118,18138538.67,0.1308,0.2394,0.2393
40-49,579,193,29555842.24,0.3333,0.3915,0.3899
50-59,186,109,16909711.79,0.5860,0.2211,0.2231
60+,105,34,5080004.37,0.3238,0.0690,0.0670



---ProductGroup---


,customers,churned,balance_leaving,churn_rate,share_of_hv_churners,share_of_balance_leaving
ProductGroup,,,,,,
1,1410,353,54089864.10,0.2504,0.7160,0.7136
2,647,72,11361245.80,0.1113,0.1460,0.1499
3+,71,68,10344227.49,0.9577,0.1379,0.1365



---ProductGroup---


,customers,churned,balance_leaving,churn_rate,share_of_hv_churners,share_of_balance_leaving
IsActiveMember,,,,,,
0,1052,320,49079810.70,0.3042,0.6491,0.6475
1,1076,173,26715526.69,0.1608,0.3509,0.3525



---ProductGroup---


,customers,churned,balance_leaving,churn_rate,share_of_hv_churners,share_of_balance_leaving
Gender,,,,,,
Female,941,266,40797543.07,0.2827,0.5396,0.5383
Male,1187,227,34997794.32,0.1912,0.4604,0.4617


In [11]:
mask = (df["Balance"] >= 100_000) & (df["IsActiveMember"] == 0)
print(value_summary(mask, "Balance >= 100k & inactive"))

{'segment': 'Balance >= 100k & inactive', 'customers': 2356, 'churned': 772, 'churn_rate': 0.3277, 'avg_balance': np.float64(132541.0), 'balance_total': np.float64(312265431.0), 'balance_leaving': np.float64(101638895.0)}


In [12]:
prio = (
    (df["Balance"] >= 100_000)
    & (df["IsActiveMember"] == 0)
    & (df["ProductGroup"] == "1")
    & (df["AgeBand"].isin(["40-49","50-59"]))
)
print(value_summary(prio, "Priority: >=100k , inactive, 1 product, 40-59"))

hv_ct = hv.pivot_table(index="Geography",columns="IsActiveMember",values="Exited",aggfunc=["size","mean"], observed=True)
hv_ct.round(4)

{'segment': 'Priority: >=100k , inactive, 1 product, 40-59', 'customers': 618, 'churned': 353, 'churn_rate': 0.5712, 'avg_balance': np.float64(132489.0), 'balance_total': np.float64(81878451.0), 'balance_leaving': np.float64(46572047.0)}


size         mean        
IsActiveMember    0    1       0       1
Geography                               
France          462  443  0.2706  0.1377
Germany         387  398  0.3618  0.1960
Spain           203  235  0.2709  0.1447

In [13]:
tot_leaving = df.loc[df["Exited"] == 1, "Balance"].sum()
prio_leaving = df.loc[prio & (df["Exited"] == 1), "Balance"].sum()
print("Priority share of all balance leaving:", round(prio_leaving / tot_leaving, 4))
print("Priority share of customers:", round(prio.mean(), 4))
print("Priority share of churners:", round(df.loc[prio, "Exited"].sum() / df["Exited"].sum(), 4))

p = df[prio]
p.groupby("Geography", observed=True)["Exited"].agg(customers="size", churn_rate="mean").round(4)

Priority share of all balance leaving: 0.2509
Priority share of customers: 0.0618
Priority share of churners: 0.1733


,customers,churn_rate
Geography,,
France,234,0.4444
Germany,282,0.7518
Spain,102,0.3627


## Findings: High-Value Churn

- **High-value group:** the top balance band (balance above roughly 128k) has 2,128 customers (21.3% of the bank) and holds 42.4% of all balance. Its churn rate is 23.17%, only slightly above the 20.37% average, but 493 churners take 75.8M with them, which is 40.8% of all balance leaving the bank (185.6M).
- **Engagement matters more than balance:** customers with balance of at least 100k who are active churn at 17.97%; inactive ones churn at 32.77% and account for 101.6M leaving (54.8% of all balance leaving) from 23.6% of customers.
- **Priority segment:** high balance, inactive, one product, age 40-59. It has 618 customers (6.2% of the bank) and a 57.12% churn rate. It loses 46.6M, which is 25.1% of all balance leaving, and contains 17.3% of all churners.
- **Germany is the hot spot:** within the priority segment, Germany churns at 75.18% (282 customers) versus France 44.44% (234) and Spain 36.27% (102).
- **Other drivers inside the high band:** age 50-59 (58.6%), women (28.3% vs 19.1% men), single-product holders (25.0%), and 3+ products (95.8%, only 71 customers).
- **Implication:** retention effort should start with inactive, single-product, middle-aged customers with large balances, especially in Germany. Re-engagement and cross-selling a second product look like the most direct levers (the 3+ product group is a separate product-design problem, not a target).